## Setup

In [ ]:
import sys 
import re

from metabo_funcs import *
%load_ext autoreload
%autoreload 2
sns.set_theme(font="Arial", style="white")
sns.set_style("white")
data_dir = Path("/Users/henrysanford/dev/test_data/macrophage/metabolomics")

AVG_INTENSITY_CUTOFF = 5000


## Whole cell metabolomics

Read data and metadata. Rename the columns for a more readable table

In [ ]:
metabo_dir = data_dir
combined_files, filtered_files, results_files, formatted_files = make_result_dirs(
    metabo_dir / "cross_replicate_analysis"
)

results_files.mkdir(exist_ok=True)
index_cols = ["Compound", "HMDB"]
metabo_df = pd.read_excel(metabo_dir / "EV2399_Report.xlsx").set_index(index_cols)
# drop pre-computed DE stats
metabo_df = metabo_df.drop(
    (
        list(metabo_df.filter(like="ANOVA").columns)
        + list(metabo_df.filter(like="log2FC").columns)
        + list(metabo_df.filter(like="p value"))
    ), axis = 1
)

# tidy column names
new_columns = []
for f in metabo_df.columns:
    match = re.search(r'D([ABCD])(M0|LPS)', f)
    if match:
        a, l = match.group(1), match.group(2)
        new_columns.append(re.sub(r'D([ABCD])(M0|LPS)', f'{l}_d{a}', f))
    else:
        new_columns.append(f)  # keep original if no match
metabo_df.columns = new_columns
donors = {"dA", "dB","dC","dD"}
conditions = {"LPS","M0"}

Filter data on intensity cutoff. Report the percentage of missing data for each sample

In [ ]:
filtered_df = filter_on_intensity_cutoff(
    metabo_df=metabo_df,
    donors=donors,
    conditions=conditions,
    index_cols=index_cols,
    output_path=filtered_files,
)

raw_signal_filename = combined_files / "combreplicates_raw_signal.csv"

filtered_df.reset_index().drop("HMDB", axis = 1).to_csv(raw_signal_filename, index = None)

missing_data = filtered_df.replace(0, np.nan).isnull()

print(
    "% missing data in samples\n",
    (missing_data.sum() * 100 / len(filtered_df)).sort_values(),
    "% missing data in metabolites\n",
    (missing_data.sum(axis=1) * 100 / len(filtered_df)).sort_values(),
)

### Data imputation

Perform Quantile Regression Imputation of Left-Censored data. Call an R script to run imputeLCMD package

In [ ]:
impute_LCMD_output_filename = combined_files / "metabolomics_log2_imputed_data.csv"

rscript_location = "Rscript"
subprocess.call(
    "{} --vanilla data_imputation.R '{}' '{}' '{}'".format(
        str(rscript_location),
        str(raw_signal_filename),
        str(impute_LCMD_output_filename),
        "Compound"
    ),
    shell=True,
)

# read results and format for analysis

impute_lcmd_results = pd.read_csv(impute_LCMD_output_filename)

np.exp2(impute_lcmd_results.set_index("Compound")).reset_index().to_csv(
    combined_files / "combreplicates_raw_signal_with_qrilc_imputation.csv"
)

Plot the distribution of signal intensity with and without data imputation

In [ ]:
imputed_data_histogram(
    index_cols,
    filtered_df,
    impute_lcmd_results,
    results_files,
    combined_files
)

In [ ]:
imputed_data = pd.read_csv(combined_files / "combreplicates_raw_signal_with_qrilc_imputation.csv", index_col=0).set_index(index_cols[0])
channel_ratio_df = calc_channel_ratio(metabo_df = imputed_data, 
                                        donors = donors, 
                                        index_cols=[index_cols[0]])
channel_ratio_df.to_csv(combined_files / "combreplicates_channel_ratio.csv")

### Differential expression analysis

In [ ]:
volcano_df, de_results = differential_expression_analysis(
    [index_cols[0]],
    results_files,
    file_name="metabolomics",
    target_name="Metabolites",
    channel_ratio_df=channel_ratio_df
)

### Principal component analysis

Run principal component analysis. Imputed data isn't helpful for this analysis, so use data before imputation

In [ ]:
pca_input = calc_channel_ratio(
    metabo_df=filtered_df.reset_index().drop("HMDB", axis=1).set_index(index_cols[0]),
    donors=donors,
    index_cols=[index_cols[0]],
)
pca_dir = results_files / "pca"
fig, loadings_df, pca_df, percent_df = get_pca_plot(
    pca_input.reset_index(), [index_cols[0]], "Metabolomics PCA", out_dir=pca_dir
)
fig

### Final data table

In [ ]:
volcano_df = volcano_df.reset_index()
de_results = [x.reset_index() for x in de_results]

# concatanate differential expression results
final_table = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on=list(volcano_df.columns[~volcano_df.columns.str.contains("vs.")]),
        how="outer",
    ),
    de_results,
)
def add_metabolite_annotation(volcano_df, index_col="Compound"):
    """merge metabolite class annotations with quantification
    returns annotated data frame
    """
    annotation = pd.read_csv(
        data_dir / "20240924_list of polar metabolites for the targeted assay_annotations_LSP_v3.csv"
    )
    annotation_index = "NAME"
    merged_df = volcano_df.merge(
        right=annotation,
        left_on=index_col,
        right_on=annotation_index,
        how="left",
    )
    merged_df[index_col] = volcano_df.reset_index()[index_col]
    return merged_df

final_table = add_metabolite_annotation(final_table)
# merge with normalized and unnormalized channel ratio measurements
channel_ratio_df_normalized = channel_ratio_df.copy()
raw_signal = filtered_df.copy()
raw_signal.columns = [f"{x}_raw-signal-intensity" for x in raw_signal.columns]
channel_ratio_df_normalized.columns = [f"{x}_cell-volume-normalized-QRILC-imputation" for x in channel_ratio_df_normalized.columns]
final_table = raw_signal.reset_index().drop("HMDB", axis = 1).merge(right=final_table, on=index_cols[0], how="outer")
final_table = channel_ratio_df_normalized.merge(right=final_table, on=index_cols[0], how="outer")

# clean up table
final_table = final_table[
    final_table.columns[~final_table.columns.str.contains("index")]
]

column_order = list(final_table.columns[~final_table.columns.str.contains("vs.|_")])
column_order.extend(list(final_table.columns[final_table.columns.str.contains("_")]))
column_order.extend(list(final_table.columns[final_table.columns.str.contains("vs.")]))
final_table = (final_table[column_order]
               .drop("NAME", axis=1)
               .set_index(["Compound", "HMDB", "KEGG", "Alias"]))

# The manuscript supplement "Data S3.xlsx" is now built cleanly (deduplicated stat block,
# automated title/description, written to the repo root) by metabolomics/build_data_s3.py:
#     conda run -n polars python metabolomics/build_data_s3.py

In [ ]:
data_dir